In [ ]:
# Fixed symbolic_tsp_pipeline function with float32-compatible gohst movement
import numpy as np

def symbolic_tsp_pipeline(cities, symbolic_field=None, entropy_field=None, steps=8, gamma=0.005):
    FIELD_SIZE = 100
    city_mass = np.mean([np.linalg.norm(cities[i] - cities[j]) for i in range(len(cities)) for j in range(i+1, len(cities))])

    symbolic_field = symbolic_field or np.zeros((FIELD_SIZE, FIELD_SIZE))
    entropy_field = entropy_field or np.zeros((FIELD_SIZE, FIELD_SIZE))

    home_city = cities[0].copy()
    ancestral_homes = [home_city.copy()]

    for _ in range(len(cities)):
        # Initialize gohsts
        NUM_GOHSTS = int(gamma * FIELD_SIZE ** 2)
        projected_gohsts = []
        for i in range(NUM_GOHSTS):
            angle = 2 * np.pi * i / NUM_GOHSTS
            direction = np.array([np.cos(angle), np.sin(angle)], dtype=np.float32)
            projected_gohsts.append({"pos": home_city.copy().astype(np.float32), "dir": direction, "dark": False, "collapsed": False})

        # Gohst movement and collapse
        for step in range(steps):
            for gohst in projected_gohsts:
                if not gohst["collapsed"] and step == steps // 2:
                    x, y = map(int, np.clip(gohst["pos"], 0, FIELD_SIZE - 1))
                    if gohst["dark"]:
                        entropy_field[x, y] += city_mass * 6
                    else:
                        symbolic_field[x, y] += city_mass
                    gohst["collapsed"] = True
                if not gohst["collapsed"]:
                    gohst["pos"] += gohst["dir"]

        # Placeholder path planner (greedy for now)
        def greedy_tsp(coords):
            n = len(coords)
            visited = [False] * n
            path = [0]
            visited[0] = True
            for _ in range(1, n):
                last = path[-1]
                next_city = min((i for i in range(n) if not visited[i]), key=lambda j: np.linalg.norm(coords[last] - coords[j]))
                path.append(next_city)
                visited[next_city] = True
            path.append(0)
            return path

        symbolic_field = symbolic_field.astype(np.float32)
        entropy_field = entropy_field.astype(np.float32)
        path = greedy_tsp(cities)
        return path, ancestral_homes
